In [1]:
# ============================================================
# Deliberative Agent AI Project Management Assistant
# by Dr. Arpit Yadav
# LangGraph + Groq + Serper + Gradio
# ============================================================

# =========================
# 1. Install dependencies
# =========================
!pip -q install langgraph langchain langchain-groq gradio requests pandas pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 2.0 MB/s eta 0:00:00


In [2]:


# =========================
# 2. Import all necessary libraries
# =========================
import os
import re
import json
import math
import textwrap
import requests
import pandas as pd
from typing import TypedDict, List, Dict, Any

import gradio as gr

from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq

In [3]:



# =========================
# 3. Give Groq API and Serper API
# =========================
# Option A: set directly
# os.environ["GROQ_API_KEY"] = "your_groq_api_key"
# os.environ["SERPER_API_KEY"] = "your_serper_api_key"

if "GROQ_API_KEY" not in os.environ:
    try:
        from getpass import getpass
        os.environ["GROQ_API_KEY"] = getpass("Enter GROQ_API_KEY: ")
    except Exception:
        pass

if "SERPER_API_KEY" not in os.environ:
    try:
        from getpass import getpass
        os.environ["SERPER_API_KEY"] = getpass("Enter SERPER_API_KEY: ")
    except Exception:
        pass

GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")
SERPER_API_KEY = os.getenv("SERPER_API_KEY", "")


Enter GROQ_API_KEY: ··········
Enter SERPER_API_KEY: ··········


In [4]:


# =========================
# 4. Select Model from groq : use llama-3.1-8b-instant
# =========================
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    api_key=GROQ_API_KEY
)

In [5]:


# ============================================================
# Deliberative Agent Logic
# ============================================================
# This agent:
# 1. Understands the project
# 2. Breaks it into phases
# 3. Plans tasks and milestones
# 4. Allocates resources
# 5. Evaluates risks and dependencies
# 6. Produces a reasoned project plan
# ============================================================

In [6]:




# =========================
# 5. LangGraph state
# =========================
class PMState(TypedDict, total=False):
    project_name: str
    project_type: str
    industry: str
    objective: str
    scope_summary: str
    team_size: int
    duration_weeks: int
    budget_level: str
    methodology: str
    priority_mode: str
    risk_tolerance: str
    complexity: str
    search_mode: str
    special_constraints: str

    project_profile: Dict[str, Any]
    work_breakdown: List[Dict[str, Any]]
    resource_plan: List[Dict[str, Any]]
    risk_register: List[Dict[str, Any]]
    milestone_plan: List[Dict[str, Any]]
    web_evidence: str
    llm_explanation: str
    final_report: str

In [7]:
# =========================
# 6. Helper functions
# =========================
def safe_int(v, default=1):
    try:
        return int(v)
    except:
        return default

def serper_search(query: str, num_results: int = 3) -> str:
    if not SERPER_API_KEY:
        return "Serper API key not available."

    url = "https://google.serper.dev/search"
    payload = json.dumps({"q": query, "num": num_results})
    headers = {
        "X-API-KEY": SERPER_API_KEY,
        "Content-Type": "application/json"
    }

    try:
        response = requests.post(url, headers=headers, data=payload, timeout=20)
        response.raise_for_status()
        data = response.json()

        snippets = []
        for item in data.get("organic", [])[:num_results]:
            title = item.get("title", "")
            snippet = item.get("snippet", "")
            link = item.get("link", "")
            snippets.append(f"- {title} | {link}\n  {snippet}")

        if not snippets:
            return "No notable public project-management evidence found."
        return "\n".join(snippets)

    except Exception as e:
        return f"Serper search failed: {str(e)}"


def estimate_role_distribution(team_size: int):
    if team_size <= 3:
        return {
            "Project Manager": 1,
            "Business/Functional Analyst": 1,
            "Execution Team": max(1, team_size - 2)
        }
    elif team_size <= 6:
        return {
            "Project Manager": 1,
            "Business/Functional Analyst": 1,
            "Tech/Delivery Lead": 1,
            "Execution Team": max(1, team_size - 3)
        }
    else:
        return {
            "Project Manager": 1,
            "Business/Functional Analyst": 1,
            "Tech/Delivery Lead": 1,
            "QA/Risk Coordinator": 1,
            "Execution Team": max(2, team_size - 4)
        }


def base_phase_template(project_type: str):
    common = [
        "Initiation",
        "Requirement Discovery",
        "Planning & Architecture",
        "Execution / Build",
        "Testing / Validation",
        "Deployment / Handover",
        "Monitoring / Closure"
    ]

    if project_type == "AI Product":
        return [
            "Initiation",
            "Business Problem Framing",
            "Data & Requirement Discovery",
            "Architecture & Tool Selection",
            "Prototype / POC Build",
            "Testing / Evaluation",
            "Deployment / Handover",
            "Monitoring / Iteration"
        ]
    elif project_type == "Software Development":
        return [
            "Initiation",
            "Requirement Discovery",
            "Architecture & Sprint Planning",
            "Development",
            "Testing / QA",
            "Deployment",
            "Closure"
        ]
    elif project_type == "Research / Innovation":
        return [
            "Initiation",
            "Literature / Benchmark Review",
            "Experiment Planning",
            "Implementation / Experimentation",
            "Analysis / Validation",
            "Documentation / Presentation",
            "Closure"
        ]
    elif project_type == "Digital Transformation":
        return [
            "Initiation",
            "Current State Assessment",
            "Future State Design",
            "Execution Planning",
            "Implementation",
            "Adoption / Change Management",
            "Closure"
        ]
    else:
        return common


def split_duration(duration_weeks: int, phases: List[str]):
    # proportional weights for deliberative planning
    weights = []
    for p in phases:
        if "Initiation" in p:
            weights.append(0.08)
        elif "Requirement" in p or "Discovery" in p or "Review" in p:
            weights.append(0.14)
        elif "Architecture" in p or "Planning" in p or "Design" in p:
            weights.append(0.16)
        elif "Build" in p or "Development" in p or "Implementation" in p or "Experimentation" in p:
            weights.append(0.28)
        elif "Testing" in p or "Validation" in p or "Analysis" in p:
            weights.append(0.16)
        elif "Deployment" in p or "Handover" in p or "Presentation" in p or "Adoption" in p:
            weights.append(0.10)
        else:
            weights.append(0.08)

    total = sum(weights)
    norm = [w / total for w in weights]
    phase_weeks = [max(1, round(duration_weeks * w)) for w in norm]

    # adjust to exact total
    diff = duration_weeks - sum(phase_weeks)
    idx = 0
    while diff != 0 and len(phase_weeks) > 0:
        if diff > 0:
            phase_weeks[idx % len(phase_weeks)] += 1
            diff -= 1
        else:
            pos = idx % len(phase_weeks)
            if phase_weeks[pos] > 1:
                phase_weeks[pos] -= 1
                diff += 1
        idx += 1

    return phase_weeks


def priority_score(task_type: str, priority_mode: str, risk_tolerance: str):
    score = 50

    if "Requirement" in task_type or "Discovery" in task_type:
        score += 15
    if "Risk" in task_type or "Testing" in task_type or "Validation" in task_type:
        score += 12
    if "Deployment" in task_type or "Handover" in task_type:
        score += 8
    if "Architecture" in task_type or "Planning" in task_type:
        score += 10

    if priority_mode == "Speed":
        if "Build" in task_type or "Development" in task_type or "Implementation" in task_type:
            score += 10
        if "Documentation" in task_type:
            score -= 8

    elif priority_mode == "Quality":
        if "Testing" in task_type or "Validation" in task_type:
            score += 12
        if "Review" in task_type:
            score += 8

    elif priority_mode == "Balanced":
        score += 5

    if risk_tolerance == "Low":
        if "Risk" in task_type or "Testing" in task_type or "Validation" in task_type:
            score += 10
    elif risk_tolerance == "High":
        if "Build" in task_type or "Implementation" in task_type:
            score += 6

    return min(score, 100)

In [8]:

# =========================
# 7. LangGraph nodes
# =========================
def profile_project_node(state: PMState) -> PMState:
    profile = {
        "project_name": state.get("project_name", "").strip(),
        "project_type": state.get("project_type", "AI Product"),
        "industry": state.get("industry", "General"),
        "objective": state.get("objective", "").strip(),
        "scope_summary": state.get("scope_summary", "").strip(),
        "team_size": safe_int(state.get("team_size", 5), 5),
        "duration_weeks": safe_int(state.get("duration_weeks", 12), 12),
        "budget_level": state.get("budget_level", "Medium"),
        "methodology": state.get("methodology", "Agile"),
        "priority_mode": state.get("priority_mode", "Balanced"),
        "risk_tolerance": state.get("risk_tolerance", "Medium"),
        "complexity": state.get("complexity", "Medium"),
        "special_constraints": state.get("special_constraints", "").strip()
    }
    return {**state, "project_profile": profile}


def build_work_breakdown_node(state: PMState) -> PMState:
    profile = state["project_profile"]
    phases = base_phase_template(profile["project_type"])
    phase_weeks = split_duration(profile["duration_weeks"], phases)

    work_breakdown = []
    current_week = 1

    for phase, weeks in zip(phases, phase_weeks):
        start_week = current_week
        end_week = current_week + weeks - 1

        # generic deliverables by phase
        if "Initiation" in phase:
            deliverables = ["Project charter", "Stakeholder map", "Success criteria"]
        elif "Requirement" in phase or "Discovery" in phase:
            deliverables = ["Requirements list", "Scope baseline", "Dependency mapping"]
        elif "Architecture" in phase or "Planning" in phase or "Design" in phase:
            deliverables = ["Architecture/design", "Sprint plan", "Risk plan"]
        elif "Build" in phase or "Development" in phase or "Implementation" in phase or "Experimentation" in phase:
            deliverables = ["Core execution output", "Progress checkpoints", "Internal review"]
        elif "Testing" in phase or "Validation" in phase or "Analysis" in phase:
            deliverables = ["QA results", "Validation findings", "Issue log"]
        elif "Deployment" in phase or "Handover" in phase or "Presentation" in phase or "Adoption" in phase:
            deliverables = ["Deployment/handover", "User training", "Go-live checklist"]
        else:
            deliverables = ["Closure note", "Lessons learned", "Final status report"]

        priority = priority_score(phase, profile["priority_mode"], profile["risk_tolerance"])

        work_breakdown.append({
            "phase": phase,
            "start_week": start_week,
            "end_week": end_week,
            "duration_weeks": weeks,
            "priority_score": priority,
            "deliverables": ", ".join(deliverables),
            "status": "Planned"
        })

        current_week = end_week + 1

    return {**state, "work_breakdown": work_breakdown}


def resource_planning_node(state: PMState) -> PMState:
    profile = state["project_profile"]
    role_map = estimate_role_distribution(profile["team_size"])

    resource_plan = []
    for role, count in role_map.items():
        if role == "Project Manager":
            responsibilities = "Schedule, stakeholder management, escalation handling"
        elif role == "Business/Functional Analyst":
            responsibilities = "Requirements, process mapping, scope alignment"
        elif role == "Tech/Delivery Lead":
            responsibilities = "Technical planning, architecture, delivery oversight"
        elif role == "QA/Risk Coordinator":
            responsibilities = "Testing, risk tracking, issue control"
        else:
            responsibilities = "Execution of project tasks and milestone delivery"

        resource_plan.append({
            "role": role,
            "count": count,
            "responsibilities": responsibilities
        })

    return {**state, "resource_plan": resource_plan}


def risk_assessment_node(state: PMState) -> PMState:
    profile = state["project_profile"]
    risks = []

    complexity = profile["complexity"]
    risk_tolerance = profile["risk_tolerance"]
    budget = profile["budget_level"]
    methodology = profile["methodology"]
    constraints = profile.get("special_constraints", "")

    risks.append({
        "risk": "Scope creep",
        "impact": "High",
        "probability": "Medium" if methodology == "Agile" else "High",
        "mitigation": "Freeze MVP scope and use change-control review"
    })

    risks.append({
        "risk": "Timeline slippage",
        "impact": "High",
        "probability": "High" if complexity in ["High", "Very High"] else "Medium",
        "mitigation": "Weekly checkpoints and milestone tracking"
    })

    risks.append({
        "risk": "Resource bottleneck",
        "impact": "Medium",
        "probability": "High" if profile["team_size"] <= 3 else "Medium",
        "mitigation": "Role clarity and workload balancing"
    })

    risks.append({
        "risk": "Quality issues",
        "impact": "High",
        "probability": "Medium",
        "mitigation": "Earlier validation and QA gates"
    })

    if budget == "Low":
        risks.append({
            "risk": "Budget overrun",
            "impact": "High",
            "probability": "High",
            "mitigation": "Prioritize essentials and phase non-critical work"
        })

    if "compliance" in constraints.lower() or "security" in constraints.lower():
        risks.append({
            "risk": "Compliance / security non-conformance",
            "impact": "High",
            "probability": "Medium",
            "mitigation": "Add review checkpoints and compliance sign-off"
        })

    if risk_tolerance == "Low":
        risks.append({
            "risk": "Decision delay due to approval dependencies",
            "impact": "Medium",
            "probability": "Medium",
            "mitigation": "Predefine approval matrix and response timelines"
        })

    return {**state, "risk_register": risks}


def milestone_node(state: PMState) -> PMState:
    wbs = state.get("work_breakdown", [])
    milestones = []

    for idx, item in enumerate(wbs, start=1):
        milestones.append({
            "milestone_id": f"M{idx}",
            "milestone_name": item["phase"],
            "target_week": item["end_week"],
            "success_metric": f"{item['phase']} deliverables completed"
        })

    return {**state, "milestone_plan": milestones}


def optional_web_search_node(state: PMState) -> PMState:
    search_mode = state.get("search_mode", "Auto")
    profile = state["project_profile"]

    if search_mode == "Off":
        return {**state, "web_evidence": "Web search disabled."}

    should_search = search_mode in ["On", "Auto"]
    if not should_search:
        return {**state, "web_evidence": "No web evidence used."}

    query = (
        f"project management best practices for {profile['project_type']} "
        f"in {profile['industry']} industry with {profile['methodology']} methodology"
    )
    evidence = serper_search(query)
    return {**state, "web_evidence": evidence}


def llm_explainer_node(state: PMState) -> PMState:
    profile = state["project_profile"]
    work_breakdown = state.get("work_breakdown", [])
    risk_register = state.get("risk_register", [])
    milestone_plan = state.get("milestone_plan", [])
    resource_plan = state.get("resource_plan", [])
    web_evidence = state.get("web_evidence", "")

    wbs_text = "\n".join([
        f"- {x['phase']} | weeks {x['start_week']}-{x['end_week']} | priority {x['priority_score']} | deliverables: {x['deliverables']}"
        for x in work_breakdown[:8]
    ])

    risk_text = "\n".join([
        f"- {r['risk']} | impact={r['impact']} | probability={r['probability']} | mitigation={r['mitigation']}"
        for r in risk_register[:6]
    ])

    resource_text = "\n".join([
        f"- {r['role']} x {r['count']} | {r['responsibilities']}"
        for r in resource_plan
    ])

    milestone_text = "\n".join([
        f"- {m['milestone_id']} {m['milestone_name']} at week {m['target_week']}"
        for m in milestone_plan
    ])

    prompt = f"""
You are a project management AI assistant.

The deliberative planner has already created the plan.
Do NOT generate a new plan from scratch.
Your role is to explain the current plan clearly and professionally.

Project profile:
{json.dumps(profile, indent=2)}

Work breakdown:
{wbs_text}

Resource plan:
{resource_text}

Risks:
{risk_text}

Milestones:
{milestone_text}

Public web evidence:
{web_evidence}

Write:
1. Executive summary
2. Why this is a deliberative plan
3. Main execution priorities
4. Biggest risks and mitigations
5. One-line PM recommendation

Keep it concise, professional, and boardroom-friendly.
"""

    try:
        resp = llm.invoke(prompt)
        explanation = resp.content if hasattr(resp, "content") else str(resp)
    except Exception as e:
        explanation = f"LLM explanation unavailable: {str(e)}"

    return {**state, "llm_explanation": explanation}


def formatter_node(state: PMState) -> PMState:
    profile = state["project_profile"]
    wbs = state.get("work_breakdown", [])
    resources = state.get("resource_plan", [])
    risks = state.get("risk_register", [])
    milestones = state.get("milestone_plan", [])
    web_evidence = state.get("web_evidence", "")
    llm_explanation = state.get("llm_explanation", "")

    wbs_lines = "\n".join([
        f"- **{x['phase']}** | Weeks {x['start_week']}-{x['end_week']} | Priority {x['priority_score']} | Deliverables: {x['deliverables']}"
        for x in wbs
    ])

    resource_lines = "\n".join([
        f"- **{r['role']}** x {r['count']} — {r['responsibilities']}"
        for r in resources
    ])

    risk_lines = "\n".join([
        f"- **{r['risk']}** | Impact: {r['impact']} | Probability: {r['probability']} | Mitigation: {r['mitigation']}"
        for r in risks
    ])

    milestone_lines = "\n".join([
        f"- **{m['milestone_id']}** {m['milestone_name']} → Week {m['target_week']} ({m['success_metric']})"
        for m in milestones
    ])

    report = f"""
# AI Project Management Assistant

## Project Summary
**Project Name:** {profile['project_name']}
**Project Type:** {profile['project_type']}
**Industry:** {profile['industry']}
**Objective:** {profile['objective']}
**Duration:** {profile['duration_weeks']} weeks
**Team Size:** {profile['team_size']}
**Methodology:** {profile['methodology']}
**Priority Mode:** {profile['priority_mode']}
**Risk Tolerance:** {profile['risk_tolerance']}
**Complexity:** {profile['complexity']}
**Budget Level:** {profile['budget_level']}

## Deliberative Work Breakdown Structure
{wbs_lines}

## Resource Plan
{resource_lines}

## Milestone Plan
{milestone_lines}

## Risk Register
{risk_lines}

## Web Evidence
{web_evidence}

## LLM Explanation
{llm_explanation}
    """.strip()

    return {**state, "final_report": report}

In [9]:








# =========================
# 8. Build LangGraph
# =========================
builder = StateGraph(PMState)

builder.add_node("profile_project", profile_project_node)
builder.add_node("build_work_breakdown", build_work_breakdown_node)
builder.add_node("resource_planning", resource_planning_node)
builder.add_node("risk_assessment", risk_assessment_node)
builder.add_node("milestone_planning", milestone_node)
builder.add_node("optional_web_search", optional_web_search_node)
builder.add_node("llm_explainer", llm_explainer_node)
builder.add_node("formatter", formatter_node)

builder.set_entry_point("profile_project")
builder.add_edge("profile_project", "build_work_breakdown")
builder.add_edge("build_work_breakdown", "resource_planning")
builder.add_edge("resource_planning", "risk_assessment")
builder.add_edge("risk_assessment", "milestone_planning")
builder.add_edge("milestone_planning", "optional_web_search")
builder.add_edge("optional_web_search", "llm_explainer")
builder.add_edge("llm_explainer", "formatter")
builder.add_edge("formatter", END)

pm_graph = builder.compile()

In [10]:



# =========================
# 9. Main function
# =========================
def run_pm_agent(
    project_name,
    project_type,
    industry,
    objective,
    scope_summary,
    team_size,
    duration_weeks,
    budget_level,
    methodology,
    priority_mode,
    risk_tolerance,
    complexity,
    search_mode,
    special_constraints
):
    state = {
        "project_name": project_name,
        "project_type": project_type,
        "industry": industry,
        "objective": objective,
        "scope_summary": scope_summary,
        "team_size": int(team_size),
        "duration_weeks": int(duration_weeks),
        "budget_level": budget_level,
        "methodology": methodology,
        "priority_mode": priority_mode,
        "risk_tolerance": risk_tolerance,
        "complexity": complexity,
        "search_mode": search_mode,
        "special_constraints": special_constraints
    }

    result = pm_graph.invoke(state)

    profile = result.get("project_profile", {})
    wbs = result.get("work_breakdown", [])
    risks = result.get("risk_register", [])
    report = result.get("final_report", "")

    wbs_df = pd.DataFrame(wbs) if wbs else pd.DataFrame(
        columns=["phase", "start_week", "end_week", "duration_weeks", "priority_score", "deliverables", "status"]
    )
    risks_df = pd.DataFrame(risks) if risks else pd.DataFrame(
        columns=["risk", "impact", "probability", "mitigation"]
    )

    total_phases = len(wbs)
    top_priority = max([x["priority_score"] for x in wbs], default=0)
    top_risk = risks[0]["risk"] if risks else "N/A"

    return report, total_phases, top_priority, top_risk, wbs_df, risks_df

In [11]:


# =========================
# 10. Example scenarios
# =========================
EXAMPLES = {
    "AI Product Launch": (
        "Enterprise HR Copilot",
        "AI Product",
        "HR Tech",
        "Build an internal AI assistant for HR workflows",
        "MVP with policy Q&A, JD drafting, resume screening support, and analytics dashboard",
        6,
        12,
        "Medium",
        "Agile",
        "Balanced",
        "Medium",
        "High",
        "Auto",
        "Data privacy, role-based access, compliance review"
    ),
    "Software Delivery Project": (
        "Customer Support Portal",
        "Software Development",
        "SaaS",
        "Build a support portal with ticketing and analytics",
        "Web app, admin panel, ticket workflows, reporting module",
        5,
        10,
        "Medium",
        "Agile",
        "Speed",
        "Medium",
        "Medium",
        "Auto",
        "Tight deadline and limited QA bandwidth"
    ),
    "Research Project": (
        "Vision RAG Research Prototype",
        "Research / Innovation",
        "AI Research",
        "Create a multimodal RAG proof of concept",
        "Ingest PDFs and images, run retrieval, answer visual questions",
        4,
        8,
        "Low",
        "Hybrid",
        "Quality",
        "Low",
        "High",
        "Auto",
        "Need evaluation metrics and presentation demo"
    ),
    "Transformation Program": (
        "Finance Process Transformation",
        "Digital Transformation",
        "BFSI",
        "Digitize and streamline finance operations",
        "Workflow redesign, dashboarding, automation pilots, user adoption",
        7,
        16,
        "High",
        "Waterfall",
        "Balanced",
        "Low",
        "High",
        "Auto",
        "Audit readiness, governance and stakeholder approvals"
    )
}

def load_example(example_name):
    return EXAMPLES[example_name]

In [ ]:




# =========================
# 11. High-level professional Gradio UI
# =========================
CUSTOM_CSS = """
body {
    background: linear-gradient(135deg, #f7fbff, #eefaf3);
}

.gradio-container {
    max-width: 1500px !important;
    margin: auto;
}

.hero-wrap {
    border-radius: 18px;
    padding: 18px 22px;
    background: linear-gradient(90deg, #eef7ff, #f6fff8);
    border: 1px solid #d9e8ff;
    box-shadow: 0 8px 24px rgba(0,0,0,0.06);
    overflow: hidden;
}

.hero-title {
    font-size: 34px;
    font-weight: 800;
    color: #184e77;
    margin-bottom: 4px;
}

.hero-subtitle {
    font-size: 14px;
    color: #4d6b86;
    margin-bottom: 12px;
}

.marquee {
    width: 100%;
    overflow: hidden;
    white-space: nowrap;
    border-radius: 10px;
    background: linear-gradient(90deg, #dff6e8, #e6f0ff);
    border: 1px solid #cfe3ff;
    padding: 10px 0;
}

.marquee span {
    display: inline-block;
    padding-left: 100%;
    animation: marquee 18s linear infinite;
    color: #0b6e4f;
    font-weight: 700;
    font-size: 18px;
}

@keyframes marquee {
    0%   { transform: translateX(0%); }
    100% { transform: translateX(-100%); }
}

.soft-card {
    border-radius: 16px !important;
    border: 1px solid #dfe9f3 !important;
    box-shadow: 0 6px 18px rgba(0,0,0,0.05) !important;
}

.footer-note {
    color: #5b7083;
    font-size: 13px;
}
"""

theme = gr.themes.Soft(
    primary_hue="blue",
    secondary_hue="green",
    neutral_hue="slate"
)

with gr.Blocks(theme=theme, css=CUSTOM_CSS, title="AI Project Management Assistant") as demo:

    gr.HTML("""
    <div class="hero-wrap">
        <div class="hero-title">AI Project Management Assistant</div>
        <div class="hero-subtitle">Deliberative Agent using LangGraph + Groq + Serper</div>
        <div class="marquee">
            <span>Deliberative Agent AI Project Management Assistant by Dr. Arpit Yadav • Deliberative Agent AI Project Management Assistant by Dr. Arpit Yadav • Deliberative Agent AI Project Management Assistant by Dr. Arpit Yadav</span>
        </div>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=5):
            gr.Markdown("### Project Input")

            example_choice = gr.Dropdown(
                choices=list(EXAMPLES.keys()),
                value="AI Product Launch",
                label="Load Example Scenario",
                interactive=True
            )

            project_name = gr.Textbox(
                label="Project Name",
                value="Enterprise HR Copilot",
                placeholder="Enter project name..."
            )

            with gr.Row():
                project_type = gr.Dropdown(
                    choices=["AI Product", "Software Development", "Research / Innovation", "Digital Transformation", "General Project"],
                    value="AI Product",
                    label="Project Type"
                )
                industry = gr.Dropdown(
                    choices=["HR Tech", "Healthcare", "BFSI", "Manufacturing", "Education", "SaaS", "AI Research", "General"],
                    value="HR Tech",
                    label="Industry"
                )

            objective = gr.Textbox(
                label="Project Objective",
                value="Build an internal AI assistant for HR workflows",
                lines=2
            )

            scope_summary = gr.Textbox(
                label="Scope Summary",
                value="MVP with policy Q&A, JD drafting, resume screening support, and analytics dashboard",
                lines=4
            )

            special_constraints = gr.Textbox(
                label="Special Constraints",
                value="Data privacy, role-based access, compliance review",
                lines=2
            )

        with gr.Column(scale=3):
            gr.Markdown("### Planning Controls")

            with gr.Row():
                team_size = gr.Slider(2, 20, value=6, step=1, label="Team Size")
                duration_weeks = gr.Slider(4, 52, value=12, step=1, label="Duration (Weeks)")

            with gr.Row():
                budget_level = gr.Dropdown(
                    choices=["Low", "Medium", "High"],
                    value="Medium",
                    label="Budget Level"
                )
                methodology = gr.Dropdown(
                    choices=["Agile", "Waterfall", "Hybrid"],
                    value="Agile",
                    label="Methodology"
                )

            with gr.Row():
                priority_mode = gr.Dropdown(
                    choices=["Speed", "Quality", "Balanced"],
                    value="Balanced",
                    label="Priority Mode"
                )
                risk_tolerance = gr.Dropdown(
                    choices=["Low", "Medium", "High"],
                    value="Medium",
                    label="Risk Tolerance"
                )

            with gr.Row():
                complexity = gr.Dropdown(
                    choices=["Low", "Medium", "High", "Very High"],
                    value="High",
                    label="Complexity"
                )
                search_mode = gr.Dropdown(
                    choices=["Auto", "On", "Off"],
                    value="Auto",
                    label="Web Search Enrichment"
                )

            run_btn = gr.Button("Run AI Project Management Assistant", variant="primary")
            clear_btn = gr.Button("Clear", variant="secondary")

            gr.Markdown("""
            **What this assistant does**
            - Profiles project context
            - Breaks work into phases
            - Plans milestones and resources
            - Builds a risk register
            - Explains the plan using Groq
            """)

    with gr.Row():
        total_phases = gr.Number(label="Total Phases", elem_classes=["soft-card"])
        top_priority = gr.Number(label="Top Priority Score", elem_classes=["soft-card"])
        top_risk = gr.Textbox(label="Top Risk", elem_classes=["soft-card"])

    with gr.Row():
        report = gr.Markdown(label="Deliberative Project Plan")
        with gr.Column():
            wbs_table = gr.Dataframe(label="Work Breakdown Structure", interactive=False)
            risk_table = gr.Dataframe(label="Risk Register", interactive=False)

    example_choice.change(
        fn=load_example,
        inputs=example_choice,
        outputs=[
            project_name, project_type, industry, objective, scope_summary,
            team_size, duration_weeks, budget_level, methodology,
            priority_mode, risk_tolerance, complexity, search_mode,
            special_constraints
        ]
    )

    run_btn.click(
        fn=run_pm_agent,
        inputs=[
            project_name, project_type, industry, objective, scope_summary,
            team_size, duration_weeks, budget_level, methodology,
            priority_mode, risk_tolerance, complexity, search_mode,
            special_constraints
        ],
        outputs=[report, total_phases, top_priority, top_risk, wbs_table, risk_table]
    )

    clear_btn.click(
        fn=lambda: (
            "",
            0,
            0,
            "",
            pd.DataFrame(columns=["phase", "start_week", "end_week", "duration_weeks", "priority_score", "deliverables", "status"]),
            pd.DataFrame(columns=["risk", "impact", "probability", "mitigation"])
        ),
        inputs=[],
        outputs=[report, total_phases, top_priority, top_risk, wbs_table, risk_table]
    )

    gr.Markdown(
        "<div class='footer-note'>Professional product-style planning demo • Deliberative Agent • LangGraph workflow • Groq explanation layer • Serper enrichment</div>"
    )

demo.launch(debug=True, share=True)

/tmp/ipykernel_6914/2544496562.py:78: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=theme, css=CUSTOM_CSS, title="AI Project Management Assistant") as demo:
/tmp/ipykernel_6914/2544496562.py:78: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=theme, css=CUSTOM_CSS, title="AI Project Management Assistant") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://64468cdf42a1786122.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
